# 00 · Definición del problema

Este notebook fija **qué se va a resolver, cómo se va a medir y qué contará como éxito** — todo
ello antes de entrenar el primer modelo.

### Nota sobre el orden de ejecución

Aunque va numerado el primero, se escribió **después** del notebook 02. No es un descuido: el
análisis exploratorio *debe* informar el planteamiento. Fijar métricas de negocio sin haber mirado
los datos produce criterios inventados.

Lo que sí es innegociable es que **no se ha entrenado ningún modelo todavía**. Los criterios de
aceptación de la sección 8 y el método de corte de la sección 9 quedan registrados aquí y no se
tocan después. Esa es la distinción que importa: el EDA informa el planteamiento, los resultados
del modelo no lo hacen.

> Dependencia: la línea base de negocio se calcula sobre `silver.bank_customers_clean`, así que en
> la primera pasada este notebook se ejecutó tras el 02.

## 1 · Contexto y problema

Una entidad bancaria con presencia en **Francia, Alemania y España** observa una pérdida sostenida
de clientes. La base analizada contiene **10.000 clientes**, de los cuáles **2.037 (20,4%) han
abandonado**.

El problema no es la tasa en sí, sino que la entidad **no sabe a quién va a perder**. Sin esa
información, las opciones son dos y ambas malas: no hacer nada, o lanzar campañas indiscriminadas
que gastan presupuesto en clientes que no pensaban irse.

### Por qué importa

Retener es sustancialmente más barato que captar. Un cliente perdido se lleva su margen recurrente
y obliga a invertir en adquisición para reemplazarlo. Con una base de 10.000 clientes y un 20,4%
de abandono, hablamos de más de dos mil relaciones que se rompen — y el banco solo se entera
cuando ya ocurrió.

### Qué cambia con este proyecto

Pasar de un diagnóstico retrospectivo («perdimos 2.037 clientes») a uno **anticipatorio**
(«estos son los clientes que probablemente perdamos, y estas son las acciones que tienen
sentido»). El objetivo no es predecir por predecir: es **priorizar un presupuesto de retención
limitado**.

## 2 · Stakeholders

| Rol | Qué necesita | Cómo lo consume |
|---|---|---|
| **Dirección de Retención** | Saber a quién contactar este mes con un presupuesto fijo | Lista priorizada por riesgo, con acción sugerida |
| **Equipo de Campañas / CRM** | Segmentos accionables, no scores abstractos | Tabla `customer_predictions` en el sistema operacional |
| **Dirección Comercial de país** | Por qué Alemania abandona al doble | Análisis por segmento y hallazgos del EDA |
| **Riesgo y Cumplimiento** | Que el modelo no discrimine ni sea inauditable | Análisis de sesgo y trazabilidad de cada predicción |
| **Equipo de Datos** | Un pipeline reproducible y mantenible | Notebooks versionados y tablas gobernadas |

El **consumidor principal es el equipo de campañas**, y eso condiciona el diseño: necesita una
lista de clientes con una acción asociada, no un informe de métricas. Por eso las predicciones
viajan de vuelta al sistema operacional en lugar de quedarse en el lakehouse.

## 3 · Variable objetivo y predictoras

**Variable objetivo:** `exited` — booleana. `True` = el cliente abandonó el banco.

Es una **clasificación binaria supervisada**. Importa señalar lo que el dataset *no* dice: no hay
fecha de abandono ni horizonte temporal. No sabemos si se fue el mes pasado o hace tres años, así
que el modelo estima una **propensión**, no un riesgo a un plazo determinado. Esa limitación se
arrastra a la interpretación y se declara en la sección 10.

### Clasificación de las variables

| Grupo | Variables | Uso |
|---|---|---|
| **Objetivo** | `exited` | Lo que se predice |
| **Predictoras — demográficas** | `age`, `gender`, `geography` | Sensibles: ver decisión de sesgo |
| **Predictoras — producto** | `num_of_products`, `has_cr_card`, `balance`, `tenure` | |
| **Predictoras — comportamiento** | `is_active_member` | **La única accionable** |
| **Predictoras — financieras** | `credit_score`, `estimated_salary` | |
| **Derivadas** | `balance_zero`, `age_group`, `products_group`, `credit_score_band` | Creadas en Silver |
| **Trazabilidad, nunca predictoras** | `customer_id` | Para devolver la predicción al CRM |
| **Excluidas** | `row_number`, `surname` | Identificador técnico y dato personal |

La distinción entre predictoras **accionables** y **no accionables** no es decorativa: la edad, el
país y el género predicen bien y no se pueden cambiar. `is_active_member` predice mejor que
cualquiera de ellas **y sí admite intervención**. El análisis prescriptivo se construye sobre esa
distinción.

## 4 · Preguntas

### Pregunta central

> **¿Qué clientes presentan mayor probabilidad de abandonar el banco, cuáles factores explican ese
> riesgo, y qué acciones de retención debería priorizar la institución?**

### Preguntas específicas

**P1 · ¿Cuál es la tasa global de abandono y qué grado de desbalance implica?**
Condiciona la elección de métricas y el tratamiento de clases.

**P2 · ¿Qué segmentos concentran el riesgo, y cuánto del abandono total explican?**
No basta con la tasa: un segmento con 90% de abandono y 30 clientes es irrelevante frente a uno
con 35% y 2.000.

**P3 · ¿Cómo varía el riesgo con la edad, la actividad, el número de productos, el saldo, la
geografía y el puntaje crediticio?**
Con atención explícita a **relaciones no monótonas**, que el EDA preliminar ya detectó.

**P4 · ¿Qué asociaciones son estadísticamente sostenibles y cuáles no permiten afirmar
causalidad?**
Incluye descartar hallazgos aparentes que resulten ser confusores.

**P5 · ¿Supera el modelo a la regla práctica que el banco podría aplicar sin ningún modelo?**
La pregunta que decide si el proyecto aporta valor. Ver sección 7.

**P6 · ¿Cambia el desempeño entre grupos de género o geografía?**
Requisito de equidad, y condición para poder desplegarlo.

**P7 · ¿Cuántos clientes de alto riesgo identifica el umbral elegido y cuánto cuesta
equivocarse?**
Traduce la matriz de confusión a dinero.

**P8 · ¿Qué tres acciones de retención se recomiendan y a qué segmento se dirige cada una?**
El entregable que el negocio realmente usa.

## 5 · Métricas técnicas

### Por qué el *accuracy* no sirve aquí

Con un 20,4% de abandono, un modelo que prediga «nadie se va» acierta el **79,6%** de las veces y
es completamente inútil: no detecta ni un solo cliente en riesgo. La celda siguiente lo comprueba
numéricamente en lugar de afirmarlo.

### Métricas adoptadas

| Métrica | Por qué |
|---|---|
| **Recall de la clase 1** | *Principal.* Un falso negativo es un cliente que se pierde sin haberlo intentado |
| **Precisión de la clase 1** | Cada falso positivo consume presupuesto de campaña |
| **F1 de la clase 1** | Resumen del equilibrio entre las dos anteriores |
| **ROC-AUC** | Capacidad de ordenar, independiente del umbral |
| **PR-AUC** | Más informativa que ROC con clases desbalanceadas |
| **Matriz de confusión** | Los cuatro números que se traducen a euros |

**El *accuracy* se reporta únicamente para documentar por qué se descarta.**

La métrica que se optimiza en el ajuste de hiperparámetros es **average precision** (área bajo la
curva precisión-recall), no *accuracy* ni ROC-AUC. Es la que mejor refleja el problema real:
ordenar bien a la minoría.

In [0]:
import pandas as pd

CATALOG      = "bank_churn"
SILVER_TABLE = f"{CATALOG}.silver.bank_customers_clean"

df = spark.table(SILVER_TABLE).toPandas()
df["exited"] = df["exited"].astype(bool)

n          = len(df)
n_churn    = int(df["exited"].sum())
tasa       = df["exited"].mean()

print(f"clientes            : {n:,}")
print(f"abandonos           : {n_churn:,}")
print(f"tasa de abandono    : {tasa:.2%}")
print(f"desbalance          : 1 a {(1 - tasa) / tasa:.1f}")

print("\n--- el problema del accuracy ---")
print(f"modelo 'nadie se va' -> accuracy {1 - tasa:.2%}, recall clase 1 {0:.0%}")
print(f"                        clientes en riesgo detectados: 0 de {n_churn:,}")

clientes            : 10,000
abandonos           : 2,037
tasa de abandono    : 20.37%
desbalance          : 1 a 3.9

--- el problema del accuracy ---
modelo 'nadie se va' -> accuracy 79.63%, recall clase 1 0%
                        clientes en riesgo detectados: 0 de 2,037


## 6 · Métrica de negocio y modelo de costes

Las métricas técnicas no se pueden llevar a un comité. Hay que traducirlas a euros, y para eso
hacen falta supuestos que el dataset no contiene.

**Se declaran explícitamente como supuestos, no como datos.** Son parametrizables: si el banco
aporta cifras reales, se sustituyen y todo el análisis se recalcula.

| Parámetro | Valor supuesto | Justificación |
|---|---|---|
| Margen anual por cliente | 200 € | Orden de magnitud típico en banca minorista |
| Horizonte de permanencia | 3 años | Sin descuento, para no añadir supuestos |
| Coste de contacto | 20 € | Gestión comercial y canal |
| Coste del incentivo | 50 € | Solo si el cliente acepta |
| Tasa de éxito de la campaña | 30% | De los que iban a irse y son contactados |
| Capacidad operativa | 800 clientes/campaña | Presupuesto fijo del equipo |

### Métrica de negocio principal

**Valor esperado neto de la campaña**: euros retenidos menos coste de contactar, para un volumen
de contactos dado.

Y una métrica secundaria, más legible para negocio: **cuántos de los clientes que iban a abandonar
se detectan dentro del cupo de 800 contactos**.

In [0]:
# --- Supuestos económicos: parámetros, no datos -----------------------------
MARGEN_ANUAL     = 200      # EUR por cliente y año
HORIZONTE_ANOS   = 3
COSTE_CONTACTO   = 20       # EUR
COSTE_INCENTIVO  = 50       # EUR, solo si acepta
TASA_EXITO       = 0.30     # de los churners contactados
CAPACIDAD        = 800      # contactos por campaña

CLV = MARGEN_ANUAL * HORIZONTE_ANOS

# Valor esperado de contactar a un cliente, según se vaya a ir o no
ganancia_si_churner    = TASA_EXITO * CLV - COSTE_CONTACTO - TASA_EXITO * COSTE_INCENTIVO
perdida_si_no_churner  = -(COSTE_CONTACTO + TASA_EXITO * COSTE_INCENTIVO)

print(f"CLV supuesto                      : {CLV:,.0f} EUR")
print(f"Valor esperado si SI se iba  (TP) : {ganancia_si_churner:+,.0f} EUR")
print(f"Valor esperado si NO se iba  (FP) : {perdida_si_no_churner:+,.0f} EUR")
print(f"Ratio beneficio/coste             : {abs(ganancia_si_churner / perdida_si_no_churner):.1f} a 1")

# Umbral que iguala el valor esperado a cero:
#   p * ganancia + (1-p) * perdida = 0
umbral_costes = -perdida_si_no_churner / (ganancia_si_churner - perdida_si_no_churner)
print(f"\nUmbral donde contactar deja de ser rentable: p = {umbral_costes:.3f}")
print(f"Muy por debajo de 0.5 -> usar 0.5 dejaría fuera clientes rentables de contactar.")

CLV supuesto                      : 600 EUR
Valor esperado si SI se iba  (TP) : +145 EUR
Valor esperado si NO se iba  (FP) : -35 EUR
Ratio beneficio/coste             : 4.1 a 1

Umbral donde contactar deja de ser rentable: p = 0.194
Muy por debajo de 0.5 -> usar 0.5 dejaria fuera clientes rentables de contactar.


## 7 · Línea base de negocio

Aquí está el criterio que decide si el proyecto tiene sentido.

El `DummyClassifier` demuestra que el *accuracy* no sirve, pero es un rival de paja. **La
competencia real del modelo es lo que el banco puede hacer gratis y en una tarde**: una regla
basada en la variable que el EDA identificó como dominante.

Si un Random Forest ajustado durante dos días no supera claramente a un `WHERE age BETWEEN 41 AND
60`, el proyecto no justifica su coste. Casi ningún trabajo de churn se somete a esta prueba —
todos se comparan contra el azar, que es un listón que cualquier cosa supera.

In [0]:
# Línea base de negocio: contactar por tramo de edad, sin modelo.
# Se calcula sobre Silver, con los cortes definitivos.
base = (df.groupby("age_group")
          .agg(clientes=("exited", "size"), abandonos=("exited", "sum"))
          .assign(tasa=lambda d: (d.abandonos / d.clientes * 100).round(1))
          .sort_values("tasa", ascending=False))

base["pct_base"]     = (base.clientes  / n       * 100).round(1)
base["pct_abandono"] = (base.abandonos / n_churn * 100).round(1)
base["lift"]         = (base.pct_abandono / base.pct_base).round(2)

print("Riesgo por tramo de edad (desde Silver):")
print(base.to_string())

# Regla candidata: los dos tramos de mayor riesgo
regla    = df["age_group"].isin(["40-49", "50-59"])
cobertura = df.loc[regla, "exited"].sum() / n_churn
volumen   = regla.mean()

print(f"\n--- REGLA BASE: contactar a los clientes de 40 a 59 años ---")
print(f"clientes contactados : {regla.sum():,} ({volumen:.1%} de la base)")
print(f"abandonos alcanzados : {int(df.loc[regla, 'exited'].sum()):,} ({cobertura:.1%} del total)")
print(f"lift                 : {cobertura / volumen:.2f}x")
print(f"precisión            : {df.loc[regla, 'exited'].mean():.1%}")

Riesgo por tramo de edad (desde Silver):
           clientes  abandonos  tasa  pct_base  pct_abandono  lift
age_group                                                         
50-59           869        487  56.0       8.7          23.9  2.75
40-49          2618        806  30.8      26.2          39.6  1.51
60+             526        147  27.9       5.3           7.2  1.36
30-39          4346        473  10.9      43.5          23.2  0.53
18-29          1641        124   7.6      16.4           6.1  0.37

--- REGLA BASE: contactar a los clientes de 40 a 59 anos ---
clientes contactados : 3,487 (34.9% de la base)
abandonos alcanzados : 1,293 (63.5% del total)
lift                 : 1.82x
precision            : 37.1%


In [0]:
# La misma regla, ajustada a la capacidad real de la campaña (800 contactos).
# Se priorizan los tramos de mayor riesgo hasta agotar el cupo.
orden = base.index.tolist()          # ya ordenado por tasa descendente
acum_clientes = acum_abandonos = 0
seleccion = []

for tramo in orden:
    fila = base.loc[tramo]
    if acum_clientes + fila.clientes <= CAPACIDAD:
        seleccion.append(tramo)
        acum_clientes   += fila.clientes
        acum_abandonos  += fila.abandonos
    else:
        restante = CAPACIDAD - acum_clientes
        if restante > 0:
            seleccion.append(f"{tramo} (parcial: {restante})")
            acum_clientes  += restante
            acum_abandonos += restante * fila.tasa / 100
        break

print(f"Con capacidad de {CAPACIDAD} contactos, la regla de edad selecciona: {seleccion}")
print(f"  contactados      : {int(acum_clientes):,}")
print(f"  abandonos captados: {int(acum_abandonos):,} de {n_churn:,} ({acum_abandonos/n_churn:.1%})")

valor_esperado = acum_abandonos * ganancia_si_churner + (acum_clientes - acum_abandonos) * perdida_si_no_churner
print(f"  valor esperado    : {valor_esperado:+,.0f} EUR")
print("\nEste es el número que el modelo debe superar.")

Con capacidad de 800 contactos, la regla de edad selecciona: ['50-59 (parcial: 800)']
  contactados      : 800
  abandonos captados: 448 de 2,037 (22.0%)
  valor esperado    : +52,640 EUR

Este es el numero que el modelo debe superar.


## 8 · Criterios de aceptación

**Registrados antes de entrenar ningún modelo.** No se modifican en función de los resultados.

### Criterio principal — superar la línea base de negocio

> Con el **mismo volumen de contactos**, el modelo debe captar **más abandonos** que la regla de
> edad calculada en la sección 7.

Es el único criterio que decide si el proyecto justifica su existencia. Los demás son condiciones
de calidad.

### Criterios técnicos

| Criterio | Umbral | Motivo |
|---|---|---|
| ROC-AUC en test | ≥ 0,75 | Capacidad de ordenación razonable |
| Recall clase 1 en el umbral elegido | ≥ 0,60 | Detectar al menos tres de cada cinco |
| Precisión clase 1 | ≥ 0,40 | Menos de eso desperdicia demasiado presupuesto |
| Superar al `DummyClassifier` en recall | obligatorio | Comprobación mínima |

### Criterios de equidad

| Criterio | Umbral |
|---|---|
| Diferencia de recall entre grupos de género | ≤ 10 puntos, o justificada explícitamente |
| Diferencia de recall entre países | ≤ 10 puntos, o justificada explícitamente |
| Coste de excluir variables sensibles | **medido y reportado**, no asumido |

### Criterios de reproducibilidad

- Semilla fija en todo proceso aleatorio (`random_state=42`)
- Split estratificado
- Todo transformador que aprenda estadísticos, dentro del `Pipeline`
- Experimentos registrados en MLflow

### Qué contaría como fracaso

Conviene decirlo también, porque un criterio que no puede fallar no es un criterio:

- Que el modelo no supere la regla de edad a igual volumen de contactos
- Que el recall se desplome en algún grupo protegido sin explicación
- Que el rendimiento dependa de una variable identificadora que se coló por error

## 9 · Método de corte de riesgo — pre-registrado

`risk_level` toma tres valores: `alto`, `medio`, `bajo`. **El método para fijar las fronteras se
decide aquí; las cifras se calculan en el notebook 05.**

### Método

| Nivel | Definición | Acción asociada |
|---|---|---|
| **alto** | Los `CAPACIDAD` clientes con mayor probabilidad predicha | Contacto proactivo con incentivo |
| **medio** | Por encima del umbral de rentabilidad de costes, fuera del cupo | Vigilancia; entra si sobra presupuesto |
| **bajo** | El resto | Sin acción |

### Por qué capacidad operativa y no un umbral fijo

Porque es como se decide de verdad. El presupuesto de campaña es fijo, y la pregunta del negocio
no es «¿quién está en riesgo?» sino **«¿a quién llamo con los recursos que tengo?»**. Un corte en
0,5 o en 0,3 es arbitrario y puede producir una lista de 3.000 personas cuando solo se puede
llamar a 800.

El umbral de rentabilidad calculado en la sección 6 sirve de **validación**: si el cupo de 800
llega a incluir clientes por debajo de ese umbral, contactarlos destruiría valor y el cupo debe
recortarse.

### Por qué se pre-registra

Si los cortes se eligen **después** de ver la distribución de scores, la tentación —consciente o
no— es elegir los que hacen quedar bien al modelo. Eso es *post-hoc rationalization*, y es uno de
los errores que un revisor con experiencia busca primero.

El enunciado lo exige de forma literal: *«establecer criterios de aceptación antes de conocer el
resultado del modelo»*.

## 10 · Supuestos y limitaciones declarados

Se declaran ahora, no al final del informe cuando ya no incomodan.

### Sobre los datos

**El dataset es sintético.** Evidencia acumulada en el EDA preliminar: `age` y `tenure`
independientes (ρ = −0,01), `tenure` y `estimated_salary` con distribución uniforme (curtosis
≈ −1,2), y `num_of_products = 4` con 100% de abandono en 60 casos. Implica un **techo estructural**:
las interacciones que un modelo puede aprender se limitan a las que el generador introdujo.

**No hay dimensión temporal.** Sin fecha de abandono ni de alta, el modelo estima propensión, no
riesgo a un plazo. No se puede decir «se irá en los próximos tres meses».

**El efecto Alemania no es explicable con estos datos.** Duplica el abandono siendo indistinguible
en todas las variables observables. `geography` funcionará como sustituto de un factor
desconocido.

**No hay catálogo de productos.** `num_of_products` es un conteo sin detalle, así que el patrón
más fuerte del dataset queda descrito pero no explicado.

### Sobre los supuestos económicos

Las seis cifras de la sección 6 son **estimaciones razonadas, no datos del banco**. Cambian el
umbral óptimo y el valor esperado, aunque no el orden de los clientes por riesgo. Se recomienda
sustituirlas por cifras reales antes de cualquier despliegue.

### Sobre la causalidad

Todo lo que se reporte serán **asociaciones**. Que la inactividad se asocie al abandono no prueba
que reactivar lo evite: podría ser síntoma y no causa — quien ya decidió irse deja de usar la
cuenta. Distinguirlo requiere un experimento (análisis de uplift), no este dataset.

### Sobre el uso previsto

El modelo se diseña para **asignar un beneficio**: una oferta de retención. Si se usara para lo
contrario —desinvertir en clientes que se van a ir— el análisis de sesgo tendría que rehacerse por
completo, porque el signo del daño se invierte.

---

## Resumen

| | |
|---|---|
| **Problema** | Clasificación binaria: predecir `exited` |
| **Objetivo real** | Priorizar un presupuesto de retención limitado |
| **Métrica principal** | Recall de la clase 1 |
| **Métrica de optimización** | Average precision |
| **Métrica de negocio** | Valor esperado neto de la campaña |
| **Rival a batir** | Regla de edad, no el azar |
| **Corte de riesgo** | Capacidad operativa, validada contra el umbral de costes |
| **Variable accionable** | `is_active_member` |

**Siguiente:** `03_eda_analisis` — formalizar con visualizaciones e inferencia los hallazgos que el
notebook 02 estableció numéricamente.